# Invoice Expense Classification

This notebook builds a simple invoice expense classifier using TF-IDF features and Logistic Regression, then exposes the model through a FastAPI `/predict` endpoint.

Supported categories:
- Logistics
- Office Supplies
- Cloud/Software
- Utilities
- Travel
- Inventory

## Install Dependencies

Run this once in a notebook environment if the required packages are not already installed.

In [1]:
# Install dependencies for a fresh notebook environment.
# In VS Code notebooks, this is usually enough to set up the runtime.
!pip install -q fastapi uvicorn scikit-learn pandas joblib nltk pytest requests

## Import Libraries

The notebook uses pandas and scikit-learn for training, plus FastAPI and joblib for the API and artifact persistence.

In [2]:
from pathlib import Path
import re
import time
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

# Optional NLTK setup for a lightweight normalization example.
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Ensure required NLTK data packages are present (download if missing).
nltk_packages = ["stopwords", "wordnet", "omw-1.4"]
for pkg in nltk_packages:
    try:
        nltk.data.find(f"corpora/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

## Sample Training Data and Preprocessing

This section uses the repository CSV for training (no sample CSV is written).

In [3]:
repo_root = Path.cwd()
data_path = repo_root / "data" / "training_data.csv"
output_dir = repo_root / "artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

# Use the existing training CSV in `data/training_data.csv` (no sample file created).
train_df = pd.read_csv(data_path, sep=None, engine="python")

# A reusable preprocessing function for inference (uses lemmatizer + stopwords).
def preprocess_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = []
    for token in text.split():
        if token not in stop_words:
            tokens.append(lemmatizer.lemmatize(token))
    return " ".join(tokens)

# Apply preprocessing consistently into a single column used later by the pipeline.
train_df["preprocessed_text"] = train_df["text"].astype(str).apply(preprocess_text)
train_df[["text", "category", "preprocessed_text"]].head(8)

,text,category,preprocessed_text
0,Filing cabinet and folders purchase,Office Supplies,filing cabinet folder purchase
1,Flight tickets Mumbai to Delhi business,Travel,flight ticket mumbai delhi business
2,Bulk purchase electronic components PCB,Inventory,bulk purchase electronic component pcb
3,Filing cabinet and folders purchase,Office Supplies,filing cabinet folder purchase
4,DHL international freight charges,Logistics,dhl international freight charge
5,Raw materials procurement steel sheets,Inventory,raw material procurement steel sheet
6,Slack Teams license renewal monthly,Cloud/Software,slack team license renewal monthly
7,Slack Teams license renewal monthly,Cloud/Software,slack team license renewal monthly


## Train / Test Split, Vectorization, Training, and Evaluation

This single cell keeps the core ML workflow together so the notebook remains compact and easy to run end to end.

In [4]:
# Prepare features and labels
X = train_df["preprocessed_text"]
y = train_df["category"]

# Split into train / test with stratification to keep class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Shapes -> X_train:", X_train.shape, "X_test:", X_test.shape)
print("Train distribution:\n", y_train.value_counts())
print("Test distribution:\n", y_test.value_counts())

# Vectorize text
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=4000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train classifier
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
start_time = time.time()
model.fit(X_train_tfidf, y_train)
training_seconds = time.time() - start_time

# Evaluate on test set
predictions = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, predictions)
report = classification_report(y_test, predictions, zero_division=0)
conf_matrix = confusion_matrix(y_test, predictions, labels=model.classes_)

print(f"Training time: {training_seconds:.2f} seconds")
print(f"Accuracy: {accuracy:.3f}")
print("\nClassification report:\n", report)
print("\nConfusion matrix (labels in model.classes_ order):\n", conf_matrix)

# Show top features per class
feature_names = vectorizer.get_feature_names_out()
coefs = model.coef_
for class_name, class_index in zip(model.classes_, range(len(model.classes_))):
    top_indices = coefs[class_index].argsort()[-8:][::-1]
    top_features = [feature_names[index] for index in top_indices]
    print(f"\nTop features for {class_name}: {top_features}")

def predict_category(text: str) -> dict:
    cleaned_text = preprocess_text(text)
    features = vectorizer.transform([cleaned_text])
    probabilities = model.predict_proba(features)[0]
    best_index = int(probabilities.argmax())
    return {
        "category": model.classes_[best_index],
        "confidence": round(float(probabilities[best_index]), 4),
    }

print("\nExample prediction 1:", predict_category("Blue Dart courier charges for warehouse delivery"))
print("Example prediction 2:", predict_category("AWS monthly cloud hosting bill"))

# Save trained artifacts to `artifacts` directory
model_path = output_dir / "model.joblib"
vectorizer_path = output_dir / "vectorizer.joblib"
joblib.dump(model, model_path)
joblib.dump(vectorizer, vectorizer_path)
print(f"Saved model -> {model_path}")
print(f"Saved vectorizer -> {vectorizer_path}")


Shapes -> X_train: (800,) X_test: (200,)
Train distribution:
 category
Logistics          145
Utilities          138
Cloud/Software     138
Office Supplies    135
Travel             130
Inventory          114
Name: count, dtype: int64
Test distribution:
 category
Logistics          36
Cloud/Software     35
Utilities          34
Office Supplies    34
Travel             32
Inventory          29
Name: count, dtype: int64
Training time: 0.05 seconds
Accuracy: 1.000

Classification report:
                  precision    recall  f1-score   support

 Cloud/Software       1.00      1.00      1.00        35
      Inventory       1.00      1.00      1.00        29
      Logistics       1.00      1.00      1.00        36
Office Supplies       1.00      1.00      1.00        34
         Travel       1.00      1.00      1.00        32
      Utilities       1.00      1.00      1.00        34

       accuracy                           1.00       200
      macro avg       1.00      1.00      1.00     

## Persist Model, FastAPI Endpoint, and Usage Examples

The next cells save the trained artifacts, define the API, and show example client requests.

In [5]:
model_path = output_dir / "model.joblib"
vectorizer_path = output_dir / "vectorizer.joblib"

loaded_model = joblib.load(model_path)
loaded_vectorizer = joblib.load(vectorizer_path)

class PredictRequest(BaseModel):
    text: str

app = FastAPI(title="Invoice Expense Classifier", version="1.0.0")

@app.post("/predict")
def predict(request: PredictRequest) -> dict:
    cleaned_text = preprocess_text(request.text)
    features = loaded_vectorizer.transform([cleaned_text])
    probabilities = loaded_model.predict_proba(features)[0]
    best_index = int(probabilities.argmax())
    return {
        "category": loaded_model.classes_[best_index],
        "confidence": round(float(probabilities[best_index]), 4),
    }

# Example API usage data.
example_input = {"text": "AWS monthly cloud hosting bill"}
print("Example /predict payload:", example_input)
print("Example /predict output:", predict(PredictRequest(**example_input)))

Example /predict payload: {'text': 'AWS monthly cloud hosting bill'}
Example /predict output: {'category': 'Cloud/Software', 'confidence': 0.8123}


In [6]:
# To run the API locally in a notebook session, uncomment the next line.
# uvicorn.run(app, host="0.0.0.0", port=8000)

print("Run the app with: uvicorn invoice_expense_classification:app --reload --port 8000")
print("Then POST JSON to /predict, for example using curl or requests.")

Run the app with: uvicorn invoice_expense_classification:app --reload --port 8000
Then POST JSON to /predict, for example using curl or requests.


## Example Requests

Use these snippets after the API is running.

In [7]:
requests_example = '''import requests

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"text": "Blue Dart courier charges for warehouse delivery"},
    timeout=30,
)
print(response.json())
'''

curl_example = '''curl -X POST "http://127.0.0.1:8000/predict" ^
  -H "Content-Type: application/json" ^
  -d "{\"text\": \"AWS monthly cloud hosting bill\"}"
'''

print("Python requests example:\n")
print(requests_example)
print("cURL example:\n")
print(curl_example)

Python requests example:

import requests

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"text": "Blue Dart courier charges for warehouse delivery"},
    timeout=30,
)
print(response.json())

cURL example:

curl -X POST "http://127.0.0.1:8000/predict" ^
  -H "Content-Type: application/json" ^
  -d "{"text": "AWS monthly cloud hosting bill"}"



## Optional Artifacts

This section writes example files for Docker, a standalone training script, unit tests, and a README snippet. Execute it if you want the notebook to generate repository assets automatically.

In [8]:
from textwrap import dedent

# Optional artifact templates. Uncomment the write calls if you want the notebook to generate them.
dockerfile_text = dedent('''
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
''').strip()

train_py_text = dedent('''
import argparse
from pathlib import Path
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import re


def preprocess_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^\\w\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="data/training_data.csv")
    parser.add_argument("--output-dir", default="artifacts")
    args = parser.parse_args()

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(args.data)
    df["clean_text"] = df["text"].apply(preprocess_text)
    X_train, X_test, y_train, y_test = train_test_split(
        df["clean_text"], df["category"], test_size=0.2, random_state=42, stratify=df["category"]
    )

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=4000)
    model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)

    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)
    model.fit(X_train_tfidf, y_train)

    predictions = model.predict(X_test_tfidf)
    print(classification_report(y_test, predictions, zero_division=0))

    joblib.dump(model, output_dir / "model.joblib")
    joblib.dump(vectorizer, output_dir / "vectorizer.joblib")


if __name__ == "__main__":
    main()
''').strip()

test_py_text = dedent('''
from fastapi.testclient import TestClient
from app import app, preprocess_text

client = TestClient(app)


def test_preprocessing():
    assert preprocess_text("AWS monthly cloud hosting bill") == "aws monthly cloud hosting bill"


def test_vectorizer_transform_shapes():
    response = client.post("/predict", json={"text": "Blue Dart courier charges for warehouse delivery"})
    assert response.status_code == 200
    assert "category" in response.json()
    assert "confidence" in response.json()


def test_predict_endpoint():
    response = client.post("/predict", json={"text": "AWS monthly cloud hosting bill"})
    assert response.status_code == 200
    payload = response.json()
    assert payload["category"] in {
        "Logistics",
        "Office Supplies",
        "Cloud/Software",
        "Utilities",
        "Travel",
        "Inventory",
    }
    assert 0.0 <= payload["confidence"] <= 1.0
''').strip()

readme_text = dedent('''
# Invoice Expense Classification

## Setup

```bash
pip install fastapi uvicorn scikit-learn pandas joblib nltk pytest requests
```

## Train

```bash
python train.py --data data/training_data.csv --output-dir artifacts
```

## Run API

```bash
uvicorn app:app --reload --port 8000
```

## Predict

```bash
curl -X POST "http://127.0.0.1:8000/predict" \
  -H "Content-Type: application/json" \
  -d '{"text": "AWS monthly cloud hosting bill"}'
```

## Docker

```bash
docker build -t invoice-classifier .
docker run -p 8000:8000 invoice-classifier
```

## Tests

```bash
pytest -q
```
''').strip()

readme_path = repo_root / "README.md"
readme_path.write_text(readme_text, encoding="utf-8")
print(f"Wrote {readme_path}")
print("Dockerfile template:\n", dockerfile_text)
print("\ntrain.py template:\n", train_py_text[:800], "...", sep="")
print("\ntests/test_api.py template:\n", test_py_text)


Wrote C:\Users\Saksham Bansal\Desktop\invoice-classifier\README.md
Dockerfile template:
 FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

train.py template:
import argparse
from pathlib import Path
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import re


def preprocess_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="data/training_data.csv")
    parser.add_argument("--output-dir", default="artifacts")
    args = parser.parse_args()

